In [3]:
import psycopg2
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
import plotly.express as px

In [ ]:
host = ""
port = ""
dbname = ""
user = ""
password = ""

In [5]:
def fetch_data_pandas(sql_query):
  try:
      connection = psycopg2.connect(
          host=host,
          port=port,
          dbname=dbname,
          user=user,
          password=password
      )

      #print(connection.get_dsn_parameters(), "\n")

      return pd.read_sql(sql_query, connection)

  except Exception as error:
      print("Error while connecting to PostgreSQL", error)

  finally:
      if (connection):
          connection.close()
          print("PostgreSQL connection is closed")

In [48]:
query = """
select 
event_date,
sum(revenue) as revenue
from mobile_game.transactions t 
group by event_date 
"""

In [49]:
df = fetch_data_pandas(query)

PostgreSQL connection is closed


In [50]:
px.line(df, x='event_date', y='revenue')

In [47]:
query_1 = """
select 
t.event_date,
ui.country,
sum(t.revenue) as revenue
from mobile_game.transactions t 
left join mobile_game.user_info ui 
on t.user_id = ui.user_id
group by t.event_date, ui.country
"""
df_1 = fetch_data_pandas(query_1)
px.line(df_1, x='event_date', y='revenue', color='country')

PostgreSQL connection is closed


In [33]:
query_2 = """
select
date(s.session_start_time) as event_date,
ui.country,
count(distinct s.user_id) as dau
from mobile_game.sessions s 
left join mobile_game.user_info ui
on s.user_id = ui.user_id
group by date(s.session_start_time), ui.country
order by event_date
"""
df_2 = fetch_data_pandas(query_2)
px.line(df_2, x='event_date', y='dau', color='country')

PostgreSQL connection is closed


In [35]:
query_3 = """
select 
start_s_date.event_date,
start_s_date.country,
sum(t.revenue) / nullif(count(distinct start_s_date.user_id), 0) as arpdau
from(
select
s.user_id,
date(session_start_time) as event_date,
ui.country
from mobile_game.sessions s 
left join mobile_game.user_info ui 
on s.user_id = ui.user_id) start_s_date
left join mobile_game.transactions t 
on start_s_date.user_id = t.user_id 
and start_s_date.event_date = t.event_date
where country = 'India'
group by start_s_date.event_date, start_s_date.country
"""
df_3 = fetch_data_pandas(query_3)
px.line(df_3, x='event_date', y='arpdau', color='country')

PostgreSQL connection is closed


In [38]:
query_4 = """
select 
t.event_date,
ui.country,
sum(t.revenue) / nullif(count(distinct t.user_id), 0) as arppu
from mobile_game.transactions t 
left join mobile_game.user_info ui on
t.user_id = ui.user_id
where country = 'India'
group by t.event_date, ui.country
"""
df_4 = fetch_data_pandas(query_4)
px.line(df_4, x='event_date', y='arppu', color='country')

PostgreSQL connection is closed


In [41]:
query_5 = """
with dau as(
select
date(s.session_start_time) as event_date,
ui.country,
count(distinct s.user_id) as dau
from mobile_game.sessions s 
left join mobile_game.user_info ui
on s.user_id = ui.user_id
group by event_date, ui.country
),
p_users as(
select
t.event_date,
ui.country,
count(distinct t.user_id) as paying_users
from mobile_game.transactions t 
left join mobile_game.user_info ui
on t.user_id = ui.user_id
group by t.event_date, ui.country
)

select 
dau.event_date,
dau.country,
case 
	when dau.dau = 0 then 0
	else coalesce(p_users.paying_users, 0)::float /dau.dau
end as daily_conversion
from dau
left join p_users
on dau.event_date = p_users.event_date
and dau.country = p_users.country
where dau.country = 'India'
"""
df_5 = fetch_data_pandas(query_5)
px.line(df_5, x='event_date', y='daily_conversion', color='country')

PostgreSQL connection is closed


In [42]:
query_6 = """
select 
date(s.session_start_time) as event_date,
ui.country,
count(distinct s.user_id) as returning_users
from mobile_game.sessions s
left join mobile_game.user_info ui 
on s.user_id = ui.user_id
where date(s.session_start_time) > ui.user_start_date
group by event_date, ui.country
"""
df_6 = fetch_data_pandas(query_6)
px.line(df_6, x='event_date', y='returning_users', color='country')

PostgreSQL connection is closed


In [81]:
query_6 = """
with touches as(
select
ut.user_id,
ut.touch_date,
ut.channel,
ui.user_start_date,
ui.country
from mobile_game.users_touches ut 
join mobile_game.user_info ui 
on ut.user_id = ui.user_id
where ut.touch_date <= ui.user_start_date
),

last_click as (
select 
user_id,
max(touch_date) as last_click_date
from touches
group by user_id
),

attributed_installs as (
select 
touches.user_id,
touches.user_start_date as install_date,
touches.country,
touches.channel
from touches 
join last_click lc
on touches.user_id = lc.user_id 
and touches.touch_date = lc.last_click_date
)

select 
install_date,
country,
channel,
count(*) as installs
from attributed_installs
group by install_date, country, channel
"""
df_6 = fetch_data_pandas(query_6)
px.line(df_6, x='install_date', y='installs', color='channel')

PostgreSQL connection is closed


In [54]:
query_7 = """
WITH returning_users AS (
    SELECT DISTINCT
        s.user_id,
        DATE(s.session_start_time) AS event_date,
        ui.country
    FROM mobile_game.sessions s
    LEFT JOIN mobile_game.user_info ui 
        ON s.user_id = ui.user_id
    WHERE DATE(s.session_start_time) > ui.user_start_date
),

returning_revenue AS (
    SELECT
        ru.event_date,
        ru.country,
        SUM(t.revenue) AS revenue_from_returning
    FROM returning_users ru
    JOIN mobile_game.transactions t
        ON ru.user_id = t.user_id
        AND ru.event_date = t.event_date
    GROUP BY ru.event_date, ru.country
)

SELECT *
FROM returning_revenue
WHERE country = 'India'
ORDER BY event_date, country;


"""
df_7 = fetch_data_pandas(query_7)
px.line(df_7, x='event_date', y='revenue_from_returning', color='country')

PostgreSQL connection is closed


In [55]:

query_8 = """
select 
event_date,
product_name,
sum(revenue) as revenue
from mobile_game.transactions t 
group by event_date, product_name 
"""
df_8 = fetch_data_pandas(query_8)
px.line(df_8, x='event_date', y='revenue', color='product_name')

PostgreSQL connection is closed


In [100]:
query_9 ="""
with touches as (
    select 
        ut.user_id,
        ut.touch_date,
        ut.channel,
        ui.user_start_date,
        ui.country
    from mobile_game.users_touches ut
    join mobile_game.user_info ui 
        on ut.user_id = ui.user_id
    where ut.touch_date <= ui.user_start_date
),
last_click as (
    select 
        user_id,
        max(touch_date) as last_click_date
    from touches
    group by user_id
),
attributed_users as (
    select 
        t.user_id,
        t.channel,
        t.user_start_date,
        t.country
    from touches t
    join last_click lc
        on t.user_id = lc.user_id 
        and t.touch_date = lc.last_click_date
    where t.channel = 'applovin'
)

select 
    au.user_start_date as install_date,
    au.country,
    au.channel,
    count(distinct au.user_id) as installs
from attributed_users au
group by au.user_start_date, au.country, au.channel
order by au.user_start_date, au.country;



"""
df_9 = fetch_data_pandas(query_9)
px.line(df_9, x='install_date', y='installs', color='country')

PostgreSQL connection is closed
